# ConsentML in action: tracking an ML experiment and revoking consent

[ConsentML](https://github.com/KaranamLokesh/consentml) is a Python library that adds
**training-data lineage** and **consent-revocation reporting** to ML pipelines.

This notebook:

1. Trains two models on a synthetic customer dataset, with lineage tracked by the `@track` decorator.
2. Processes a customer's *right-to-be-forgotten* request with `revoke()`.
3. Shows the remediation recommendation change after the model is retrained without that customer.
4. Verifies the tamper-evident audit log — and demonstrates that tampering is detected.

**Setup:** just run the next cell.

- **Locally** (in the repo's virtualenv, after `pip install -e .`): it imports what's already installed — no upload needed.
- **In Google Colab:** it prompts you to upload `consentml-0.1.0.dev0-py3-none-any.whl` — grab it from the repo's `dist/` folder on your machine. The cell verifies the file's checksum, so a corrupt upload fails loudly instead of erroring deep inside pip.

In [ ]:
# Works both locally (editable install already present) and in Google Colab.
import hashlib
import sys

WHEEL_SHA256 = "4d15a297255cb846fb053c14f319ee2aad148657865607fed68e6d37a6736dbc"

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Upload consentml-0.1.0.dev0-py3-none-any.whl (from the repo's dist/ folder):")
    uploaded = files.upload()
    wheel_name = next(iter(uploaded))
    digest = hashlib.sha256(uploaded[wheel_name]).hexdigest()
    assert digest == WHEEL_SHA256, (
        f"Uploaded wheel is corrupt (sha256 {digest[:16]}..., expected "
        f"{WHEEL_SHA256[:16]}...). Re-download from dist/ and try again."
    )
    !pip install -q --force-reinstall "{wheel_name}"
    for mod in [m for m in sys.modules if m == "consentml" or m.startswith("consentml.")]:
        sys.modules.pop(mod)

import consentml
print("consentml", consentml.__version__, "ready")

## 1. A synthetic customer dataset

200 customers with spend, support and tenure features, and a churn label.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

DB = "lineage.db"
Path(DB).unlink(missing_ok=True)  # start fresh so the notebook is re-runnable

n = 200
customers = pd.DataFrame(
    {
        "email": [f"user{i:03d}@example.com" for i in range(n)],
        "monthly_spend": rng.gamma(2.0, 40.0, n).round(2),
        "support_tickets": rng.poisson(1.5, n),
        "tenure_months": rng.integers(1, 60, n),
    }
)
customers["churned"] = (
    (customers["support_tickets"] > 2) & (customers["tenure_months"] < 24)
).astype(int)
customers.head()

## 2. Train models with lineage tracking

One decorator on the training function is the whole integration. ConsentML records which
subjects' data went into the run, hashes the trained model, and appends to the audit log —
subject IDs are stored as SHA-256 hashes, never raw emails.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from consentml import track

FEATURES = ["monthly_spend", "support_tickets", "tenure_months"]


@track(
    data_source="warehouse://demo/customers",
    subject_id_col="email",
    model_name="churn_predictor",
    db_path=DB,
)
def train_churn_model(df):
    model = RandomForestClassifier(n_estimators=50, random_state=0)
    model.fit(df[FEATURES], df["churned"])
    return model


churn_model = train_churn_model(customers)
print("churn_predictor trained on", len(customers), "customers,",
      "accuracy:", round(churn_model.score(customers[FEATURES], customers["churned"]), 3))

In [ ]:
upsell_df = customers.sample(120, random_state=1).copy()
upsell_df["will_upsell"] = (upsell_df["monthly_spend"] > 80).astype(int)


@track(
    data_source="warehouse://demo/customers",
    subject_id_col="email",
    model_name="upsell_scorer",
    db_path=DB,
)
def train_upsell_model(df):
    model = LogisticRegression(max_iter=500)
    model.fit(df[FEATURES], df["will_upsell"])
    return model


upsell_model = train_upsell_model(upsell_df)
print("upsell_scorer trained on", len(upsell_df), "customers")

## 3. What did ConsentML record?

A local SQLite lineage store: one row per training run, one index row per subject.

In [ ]:
import sqlite3

conn = sqlite3.connect(DB)
runs = pd.read_sql(
    "SELECT model_name, n_subjects, substr(model_hash, 1, 12) AS model_hash, started_at "
    "FROM training_runs", conn)
n_index = pd.read_sql("SELECT count(*) AS subject_index_rows FROM subject_index", conn)
conn.close()

print(runs.to_string(index=False))
print()
print(n_index.to_string(index=False))

## 4. A customer revokes consent

We pick a customer who is in **both** training sets. `revoke()` reports every model trained
on their data, with a per-model recommendation — it never deletes or modifies anything;
the operator stays in control.

In [ ]:
import json

from consentml import revoke

revoked_email = upsell_df["email"].iloc[0]  # present in both models' training data
print("Customer revoking consent:", revoked_email)
print()

report = revoke(subject_id=revoked_email, db_path=DB)
print(json.dumps(report.to_dict(), indent=2))

Both models are flagged `retrain`: the customer's data is in the **latest** run of each.

## 5. Remediate, then check again

Retrain the churn model *without* the revoked customer, then re-run the check
(`dry_run=True` reports without recording a second revocation event).

In [ ]:
remaining = customers[customers["email"] != revoked_email]
churn_model_v2 = train_churn_model(remaining)
print("churn_predictor retrained on", len(remaining), "customers")
print()

report2 = revoke(subject_id=revoked_email, db_path=DB, dry_run=True)
for action in report2.recommended_actions:
    print(action)

The recommendation for `churn_predictor` flipped to `review` — the customer's data only
appears in a *superseded* run, so the operator just needs to confirm the old artifact is no
longer deployed. `upsell_scorer` still needs a retrain.

## 6. Verifying the audit trail

Every event is hash-chained, like a mini certificate-transparency log. `verify_audit_log()`
checks three things:

1. each entry still hashes to its recorded value,
2. the chain links correctly, and
3. the log still agrees with the live tables.

That third check is the one a hash chain **cannot** do on its own — and it's the one that
catches the attack that actually matters here.

In [ ]:
from consentml import verify_audit_log

report = verify_audit_log(db_path=DB)
print("ok:      ", report.ok)
print("entries: ", report.n_entries)
print("head:    ", report.head_hash)

# Record this head hash somewhere outside the database -- CI logs, a separate
# system, a printed compliance record. A hash chain can't detect an attacker
# who rewrites the whole log from genesis and recomputes every hash; an
# external anchor can.
anchor = report.head_hash

### The attack a hash chain can't see

Suppose someone wants to hide that our revoked customer was ever in the training data.
They don't touch the audit log at all — they just delete the customer's row from
`subject_index`. The chain stays perfectly intact.

In [ ]:
import sqlite3

from consentml.hashing import hash_subject_id

# subject_index holds integer foreign keys, so the attacker resolves the
# customer's key through the subjects table to find the rows to delete.
conn = sqlite3.connect(DB)
conn.execute(
    "DELETE FROM subject_index WHERE subject_pk = "
    "(SELECT subject_pk FROM subjects WHERE subject_key = ?)",
    (hash_subject_id(revoked_email),),
)
conn.commit()
conn.close()

report = verify_audit_log(db_path=DB)
print("ok:", report.ok)
for f in report.findings:
    print(f"  [{f.code}] {f.detail}")


No `entry_hash_mismatch`, no `broken_link` — the hash chain is untouched and would have
reported everything fine. What gives the deletion away is `subject_count_mismatch`: the
audit payload says the run covered N subjects, and `subject_index` now holds fewer.

The count is compared against the **hash-protected audit payload**, not against
`training_runs.n_subjects` — that column is editable, so an attacker could just edit it to
match. Now let's tamper with the log itself.

In [ ]:
conn = sqlite3.connect(DB)
conn.execute(
    "UPDATE audit_log SET payload = replace(payload, 'churn_predictor', 'other_model') "
    "WHERE id = 1"
)
conn.commit()
conn.close()

report = verify_audit_log(db_path=DB)
print("ok:", report.ok)
for f in report.findings:
    where = f"entry {f.entry_id}" if f.entry_id is not None else "tables"
    print(f"  [{f.code}] {where}: {f.detail}")

assert not report.ok, "tampering should have been detected!"
print("\nBoth the edit and the earlier deletion are reported.")

Note that editing one entry produced exactly one finding, not a cascade. Each entry is
hashed from its own stored fields, so a single tampered row doesn't invalidate everything
after it — you get told precisely what changed.

## 7. The same workflow from the CLI

In [ ]:
!consentml revoke --subject-id user007@example.com --db lineage.db --dry-run --json

In [ ]:
# Exit codes: 0 clean, 1 problems found, 2 database unreadable -- so this can gate CI.
!consentml verify --db lineage.db; echo "exit=$?"

## Wrap-up

- `@track` gave us lineage **by construction** — no pipeline rework.
- `revoke()` answered "*which models learned from this person?*" with an auditable report.
- `verify_audit_log()` proves the compliance trail hasn't been edited — including the
  deletions a hash chain alone would miss.

ConsentML is a lineage and reporting tool, not a deletion tool: it tells you which models a
user touched, and records what you decided to do about it. MIT-licensed, local-only, zero
cloud dependencies.